# C4 Notebook 04 — ERA5-Land hourly → 14 × 4 weather sequences

**FloodRisk Sri Lanka | J26-DS-314 | PP1 retrospective hindcast**

This notebook uses **only** `ECMWF/ERA5_LAND/HOURLY` for the C4 weather branch. It reads Notebook 01's **target-free** manifest and existing irregular upstream **candidate** polygons; it never opens future target water-level values or modifies shared sources.

**Workflow:** verify the source and 09:00 cutoff → retrieve a **real 14-day Holombuwa pilot** → inspect units/coverage/QA → explicitly authorize cached catchment-day extraction → assemble all **1,429 station-day** sequences locally → save aligned `.npz`, daily CSV, QA CSV and manifest. Successful catchment-days are reused on reruns.

**Important timeline:** Forecast origin is **09:00 Asia/Colombo = 03:30 UTC**. The newest fully eligible ERA5-Land hourly validity time is **03:00 UTC = 08:30 local**. Thus each weather row covers the 24 hourly validity timestamps ending at **08:30 local**, not 09:00. The source is historical **reanalysis**; the experiment does *not* claim that ERA5-Land was published by 09:00 on the original date.

**One sequence:** 14 rows × `[precipitation_mm_sum, temperature_C_mean, dewpoint_C_mean, surface_pressure_hPa_mean]`, earliest day first, newest day last. Precipitation uses `total_precipitation_hourly` (already an hourly increment), **not** the cumulative `total_precipitation` band. Units and source: [ERA5-Land Hourly Earth Engine catalogue](https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_HOURLY). We use the product's native ~11 km grid and Earth Engine's fractional-overlap-weighted region mean; **no invented fine-resolution pixels**.

**Be aware:** At this ~11 km scale, small catchments may be represented by very few land-grid cells. Polygon QA status is preserved rather than considered proof of hydrological accuracy. The output must remain local/private until data-sharing terms are verified.

In [11]:
# Cell 1 — Imports, paths and explicit retrieval settings 
from pathlib import Path
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
import hashlib, io, json, math, os, tempfile, time, warnings
import numpy as np
import pandas as pd
from shapely.geometry import shape


REPO_ROOT_OVERRIDE = None  
EE_PROJECT = None  
ASSET = "ECMWF/ERA5_LAND/HOURLY"
SOURCE_VERSION = "C4_WEATHER_ERA5LAND_V1_0830_24H_NATIVE_WEIGHTS"
SOURCE_SCALE_M = 11132  # nominal ERA5-Land native pixel size, NOT an upsampled output resolution
CHUNK_DAYS = 5          # one server request per consecutive group of up to five catchment-days
PILOT_STATION_ID = "holombuwa"
RUN_BULK = True
MAX_NEW_STATIONS_PER_RUN = 5 # conservative quota/resume guard; rerun Cell 9 until complete
SLST = ZoneInfo('Asia/Colombo')
UTC = timezone.utc
VARIABLES = ["precipitation_mm", "temperature_C", "dewpoint_C", "surface_pressure_hPa"]
GEE_BANDS = ["total_precipitation_hourly", "temperature_2m", "dewpoint_temperature_2m", "surface_pressure"]


def find_repo_root():
    if REPO_ROOT_OVERRIDE:
        p = Path(REPO_ROOT_OVERRIDE).expanduser().resolve()
        if (p/'c4_fusion').is_dir() and (p/'shared'/'spatial'/'catchments').is_dir(): return p
        raise FileNotFoundError(f'Invalid REPO_ROOT_OVERRIDE: {p}')
    for base in [Path.cwd().resolve(), Path.cwd().resolve().parent]:
        for p in [base, *base.parents]:
            if (p/'c4_fusion').is_dir() and (p/'shared'/'spatial'/'catchments').is_dir(): return p
    raise FileNotFoundError('Repository not found. Set REPO_ROOT_OVERRIDE to J26-DS-314.')

ROOT = find_repo_root()
INDEX_PATH = ROOT/'data'/'processed'/'c4'/'manifests'/'c4_sample_index_1429.csv'
WEATHER_DIR = ROOT/'data'/'processed'/'c4'/'weather'
CACHE_DIR = WEATHER_DIR/'cache'/'daily'
CATCHMENTS_ROOT = ROOT/'shared'/'spatial'/'catchments'
if not INDEX_PATH.is_file():
    raise FileNotFoundError(f'Notebook 01 target-free index missing: {INDEX_PATH}. Finish Notebook 01 first.')
WEATHER_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print('Repository:',ROOT)
print('Source:', ASSET)
print('Private outputs:',WEATHER_DIR)
print('RUN_BULK:',RUN_BULK)

Repository: C:\Users\ASUS\projects\rp_project\J26-DS-314
Source: ECMWF/ERA5_LAND/HOURLY
Private outputs: C:\Users\ASUS\projects\rp_project\J26-DS-314\data\processed\c4\weather
RUN_BULK: True


## Phase 0 — Confirm the **source contract**, polygon paths and cutoff timeline before downloading anything

In [2]:
# Cell 2 — Read and validate Notebook 01's target-free index
index = pd.read_csv(INDEX_PATH, encoding='utf-8-sig', low_memory=False)
required = {'record_id','station_id','forecast_date','split','hindcast_origin_0900_local','polygon_repo_path'}
assert required.issubset(index.columns), f'Missing Notebook 01 manifest columns: {sorted(required-set(index.columns))}'
assert not {'target_stage_next_day_m','target_delta_stage_m','stage_at_issue_m'}.intersection(index.columns), 'This notebook must use only the target-free retrieval index.'
assert len(index)==1429 and index.record_id.is_unique
assert index.station_id.nunique()==39
assert index.split.value_counts().to_dict()=={'TRAIN':809,'VALIDATION':269,'TEST':351}, index.split.value_counts().to_dict()
index['forecast_day'] = pd.to_datetime(index.forecast_date, format='%Y-%m-%d', errors='raise').dt.normalize()
assert index['forecast_day'].notna().all()

# Verify simulated 09:00 Asia/Colombo origin for every day, without relying on source 09:30 report time.
for r in index.itertuples(index=False):
    origin = pd.Timestamp(str(r.hindcast_origin_0900_local).replace(' Asia/Colombo',''))
    expected = r.forecast_day + pd.Timedelta(hours=9)
    assert origin == expected, f'Bad simulated origin for {r.record_id}: {origin} != {expected}'

# Never use an arbitrary public point or rectangular bbox in place of the candidate upstream polygon.
station_polygons = {}
polygon_sha = {}
for sid, group in index.groupby('station_id', sort=True):
    paths = group.polygon_repo_path.dropna().unique()
    assert len(paths)==1, f'Conflicting polygon paths: {sid}'
    rel = str(paths[0]).replace(chr(92), '/')
    assert rel.startswith('shared/spatial/catchments/') and rel.endswith('_catchment_polygon_CANDIDATE.geojson'), (sid,rel)
    path = (ROOT/rel).resolve()
    assert path.is_relative_to(CATCHMENTS_ROOT.resolve()), f'Polygon path escaped shared catchment folder: {path}'
    assert path.is_file(), f'Missing polygon for {sid}: {path}'
    doc = json.loads(path.read_text(encoding='utf-8'))
    if doc.get('type') == 'FeatureCollection':
        feats=doc.get('features',[])
        assert len(feats)==1, f'Expected exactly one catchment feature: {sid}'
        feat=feats[0]
    else:
        feat=doc
    polygon = shape(feat['geometry'])
    assert polygon.geom_type in ('Polygon','MultiPolygon') and polygon.is_valid and not polygon.is_empty and polygon.area>0, sid
    station_polygons[sid] = feat['geometry']
    polygon_sha[sid] = hashlib.sha256(path.read_bytes()).hexdigest()

expected_days = {}
for sid, g in index.groupby('station_id',sort=True):
    dayset = set()
    for day in g['forecast_day']:
        dayset.update((day-pd.Timedelta(days=d)).date().isoformat() for d in range(14))
    expected_days[sid] = sorted(dayset)

def hourly_end_utc(forecast_day):
    # ERA5-Land hourly validity at 03:00 UTC is 08:30 in Asia/Colombo.
    day = pd.Timestamp(forecast_day).date()
    return datetime(day.year,day.month,day.day,3,0,tzinfo=UTC)

def source_window_for_day(day):
    end = hourly_end_utc(day)
    # Validity timestamps inclusive: previous-day 04:00 through current-day 03:00 UTC -> 24 hourly images.
    start = end - timedelta(hours=23)
    return start, end + timedelta(hours=1)

def days_for_forecast(forecast_day):
    day = pd.Timestamp(forecast_day)
    return [(day-pd.Timedelta(days=d)).date().isoformat() for d in range(13,-1,-1)]

example = pd.Timestamp('2025-11-17')
s,e = source_window_for_day(example)
assert s.isoformat()=='2025-11-16T04:00:00+00:00'
assert (e-timedelta(hours=1)).isoformat()=='2025-11-17T03:00:00+00:00'
assert (e-s).total_seconds()==24*3600
assert (e-timedelta(hours=1)).astimezone(SLST).strftime('%H:%M')=='08:30'
assert len(days_for_forecast(example))==14
print('PASS: 1,429 unique target-free rows across 39 labelled gauges.')
print('Unique required catchment-days:',sum(map(len,expected_days.values())))
print('Earliest/last required 24-hour period:', min(v[0] for v in expected_days.values()),'→',max(v[-1] for v in expected_days.values()))
print('Pilot:',PILOT_STATION_ID, '| first labelled forecast day:',index.loc[index.station_id==PILOT_STATION_ID,'forecast_day'].min().date())
print('Example first forecast, newest admissible ERA5-Land timestamp:', (e-timedelta(hours=1)).isoformat(), 'UTC')

PASS: 1,429 unique target-free rows across 39 labelled gauges.
Unique required catchment-days: 2026
Earliest/last required 24-hour period: 2025-11-04 → 2025-12-25
Pilot: holombuwa | first labelled forecast day: 2025-11-17
Example first forecast, newest admissible ERA5-Land timestamp: 2025-11-17T03:00:00+00:00 UTC


In [3]:
# Cell 3 — Initialize Earth Engine and verify the exact source bands 
import ee
try:
    ee.Initialize(project=EE_PROJECT) if EE_PROJECT else ee.Initialize()
except Exception as err:
    raise RuntimeError('Earth Engine initialization failed. Authenticate once using ee.Authenticate(), then rerun Cell 3; set EE_PROJECT if required.') from err
source = ee.ImageCollection(ASSET)
probe = source.filterDate('2025-11-15T00:00:00','2025-11-16T00:00:00')
assert probe.size().getInfo()==24, 'ERA5-Land source is missing hourly records in the probe day.'
actual_bands = set(ee.Image(probe.first()).bandNames().getInfo())
assert set(GEE_BANDS).issubset(actual_bands), f'Missing required ERA5-Land bands: {set(GEE_BANDS)-actual_bands}'
print('PASS: ERA5-Land hourly product and exact four source bands are available.')
print('Precipitation band:',GEE_BANDS[0], '(hourly increment, metres → mm).')

PASS: ERA5-Land hourly product and exact four source bands are available.
Precipitation band: total_precipitation_hourly (hourly increment, metres → mm).


## Phase 1 — Real one-station **14-day** pilot 

In [4]:
# Cell 4 — GEE daily reducer: 24 hourly images -> four native-grid polygon statistics
# Each chunk performs ONE native-grid spatial mean across a stack of dated daily bands.
# This avoids uploading/downloaded raw hourly rasters and avoids per-row EE requests.
# The mean reducer uses Earth Engine's approximate fractional pixel overlap weights.

def iso_ee(dt):
    return dt.astimezone(UTC).strftime('%Y-%m-%dT%H:%M:%S')

def key_for_day(day):
    return 'd'+day.replace('-','')

def gee_get_chunk(sid, days):
    assert 1 <= len(days) <= CHUNK_DAYS
    assert days == sorted(days) and len(days)==len(set(days))
    g = ee.Geometry(station_polygons[sid])
    full_start, _ = source_window_for_day(days[0])
    _, full_stop = source_window_for_day(days[-1])
    bounded = source.filterDate(iso_ee(full_start),iso_ee(full_stop)).select(GEE_BANDS)
    projection = ee.Image(source.first()).select('temperature_2m').projection()
    stacked_images=[]
    hourly_count_dict={}
    for day in days:
        start, stop = source_window_for_day(day)
        hours = bounded.filterDate(iso_ee(start),iso_ee(stop))
        prefix = key_for_day(day)
        hourly_count_dict[prefix+'_hour_count'] = hours.size()
        # Source *increments* summed over 24 validity times, NOT cumulative 'total_precipitation'.
        rain = hours.select('total_precipitation_hourly').sum().multiply(1000).rename(prefix+'_precipitation_mm')
        temp = hours.select('temperature_2m').mean().subtract(273.15).rename(prefix+'_temperature_C')
        dew = hours.select('dewpoint_temperature_2m').mean().subtract(273.15).rename(prefix+'_dewpoint_C')
        press = hours.select('surface_pressure').mean().divide(100).rename(prefix+'_surface_pressure_hPa')
        valid = (hours.select('total_precipitation_hourly').count().eq(24)
            .And(hours.select('temperature_2m').count().eq(24))
            .And(hours.select('dewpoint_temperature_2m').count().eq(24))
            .And(hours.select('surface_pressure').count().eq(24)))
        values = ee.Image.cat([rain,temp,dew,press]).updateMask(valid)
        # Native-grid fraction with all four source fields present for every one of the 24 hours.
        coverage = valid.unmask(0).rename(prefix+'_all24_coverage_fraction')
        # Diagnostic only: substantial negative *hourly increment* can reflect reanalysis artifact.
        negative_any = (hours.select('total_precipitation_hourly')
            .map(lambda im: ee.Image(im).lt(-0.00001)) # < -0.01 mm per hour
            .max().unmask(0).rename(prefix+'_negative_rain_pixel_fraction'))
        stacked_images.extend([values,coverage,negative_any])
    stacked = ee.Image.cat(stacked_images)
    results = stacked.reduceRegion(reducer=ee.Reducer.mean(),geometry=g,
        crs=projection,scale=SOURCE_SCALE_M,maxPixels=10000000,tileScale=2,bestEffort=False)
    # All counts and all polygon means are retrieved in one getInfo call per small chunk.
    remote = ee.Dictionary(results).combine(ee.Dictionary(hourly_count_dict),overwrite=True).getInfo()
    assert isinstance(remote,dict), 'Earth Engine returned an unexpected response.'
    records=[]
    for day in days:
        k=key_for_day(day)
        row={'station_id':sid,'weather_day':day,'hour_count':remote.get(k+'_hour_count'),
            'window_start_utc':iso_ee(source_window_for_day(day)[0]),
            'window_last_hour_utc':iso_ee(source_window_for_day(day)[1]-timedelta(hours=1)),
            'valid_24h_grid_coverage_fraction':remote.get(k+'_all24_coverage_fraction'),
            'negative_rain_pixel_fraction':remote.get(k+'_negative_rain_pixel_fraction')}
        for variable in VARIABLES:
            row[variable]=remote.get(k+'_'+variable)
        records.append(row)
    return records

In [5]:
# Cell 5 — Deterministic per-station/day cache; never silently invent missing weather
CONFIG = {
 'asset':ASSET, 'source_version':SOURCE_VERSION, 'scale_m':SOURCE_SCALE_M,
 'precip_band':GEE_BANDS[0], 'all_bands':GEE_BANDS,
 'window_hours':24, 'window_end_utc':'03:00', 'origin_local':'09:00 Asia/Colombo',
 'native_crs': 'native image projection, at nominal 11132 m',
 'aggregation':'per-source-pixel hourly precipitation sum, hourly temperature/dewpoint/pressure means; native-grid fractional-overlap weighted polygon means',
 'negative_hourly_threshold_m':-0.00001,
 'region':'full irregular source polygon; no public point or bbox substitution'
}

def fingerprint_for(sid):
    b=json.dumps({'config':CONFIG, 'polygon_sha256':polygon_sha[sid]},sort_keys=True,separators=(',',':')).encode()
    return hashlib.sha256(b).hexdigest()

def cache_path(sid,day): return CACHE_DIR/sid/f'{day}.json'

def read_cached(sid,day):
    p=cache_path(sid,day)
    if not p.is_file(): return None
    doc=json.loads(p.read_text(encoding='utf-8'))
    if doc.get('cache_key')!=fingerprint_for(sid):
        raise RuntimeError(f'Outdated weather cache for {sid}/{day}: {p}. Source or polygon changed. Move old cache aside before refetching.')
    assert doc.get('station_id')==sid and doc.get('weather_day')==day, 'Corrupted cache identity'
    validate_weather_record(doc)
    return doc

def save_if_changed(path,contents):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if isinstance(contents,str): contents=contents.encode('utf-8')
    if path.exists() and path.read_bytes()==contents: return 'UNCHANGED'
    fd,temp=tempfile.mkstemp(prefix='.tmp_',dir=str(path.parent))
    try:
        with os.fdopen(fd,'wb') as f: f.write(contents); f.flush();os.fsync(f.fileno())
        os.replace(temp,path)
    finally:
        if os.path.exists(temp):os.unlink(temp)
    return 'SAVED'

def validate_weather_record(rec):
    assert int(rec.get('hour_count',-1))==24, f'Expected 24 hourly ERA5-Land images, got {rec.get("hour_count")} for {rec.get("station_id")}/{rec.get("weather_day")}'
    for feature in VARIABLES:
        x=rec.get(feature)
        assert x is not None and math.isfinite(float(x)), f'Missing/nonfinite {feature}: {rec.get("station_id")}/{rec.get("weather_day")}'
    rain=float(rec['precipitation_mm']); temp=float(rec['temperature_C']); dew=float(rec['dewpoint_C']); press=float(rec['surface_pressure_hPa'])
    assert rain >= -0.01, f'Significant negative daily precipitation ({rain} mm). Investigate band/ERA5-Land corrections; not cached.'
    assert -35<=temp<=65 and -45<=dew<=65, f'Implausible temperature: {temp}, {dew}'
    assert 400<=press<=1100, f'Implausible surface pressure: {press}'
    coverage=rec.get('valid_24h_grid_coverage_fraction')
    assert coverage is not None and 0<float(coverage)<=1.000001, f'No valid ERA5-Land land-grid coverage: {coverage}'
    negative=rec.get('negative_rain_pixel_fraction')
    assert negative is not None and 0<=float(negative)<=1.000001
    # Quality observations remain explicit; do not silently mark the whole sample rejected.
    return True

def get_weather_days(sid, requested_days, *, allow_remote=False, verbose=True):
    days=sorted(set(requested_days))
    assert sid in station_polygons
    assert set(days).issubset(set(expected_days[sid])), f'Unexpected days requested for {sid}'
    result={}; missing=[]
    for day in days:
        cached=read_cached(sid,day)
        if cached is not None: result[day]=cached
        else: missing.append(day)
    if missing and not allow_remote:
        raise RuntimeError(f'{sid}: {len(missing)} weather days missing locally. Enable deliberate pilot/bulk retrieval. First: {missing[:3]}')
    remote_calls=0
    for at in range(0,len(missing),CHUNK_DAYS):
        chunk=missing[at:at+CHUNK_DAYS]
        if verbose: print(f'{sid}: ERA5-Land query {chunk[0]} through {chunk[-1]} ({len(chunk)} catchment-days)')
        response=gee_get_chunk(sid,chunk)
        assert len(response)==len(chunk)
        # Validate every remote record BEFORE caching any of this chunk.
        for got,day in zip(response,chunk):
            assert got['station_id']==sid and got['weather_day']==day
            validate_weather_record(got)
        for got,day in zip(response,chunk):
            got['cache_key']=fingerprint_for(sid)
            got['source_version']=SOURCE_VERSION
            save_if_changed(cache_path(sid,day),json.dumps(got,sort_keys=True,indent=2)+'\n')
            result[day]=got
        remote_calls+=1
    ordered=[result[d] for d in requested_days]
    if verbose: print(f'{sid}: {len(ordered)} valid days; reused {len(days)-len(missing)} cached days; new server calls {remote_calls}.')
    return ordered,remote_calls

In [6]:
# Cell 6 — Real 14-day Holombuwa pilot 
pilot_origin = index.loc[index.station_id==PILOT_STATION_ID,'forecast_day'].min()
assert pd.notna(pilot_origin), 'Pilot station has no labelled forecast origin.'
pilot_days=days_for_forecast(pilot_origin)
pilot_rows,pilot_calls=get_weather_days(PILOT_STATION_ID,pilot_days,allow_remote=True)
pilot=pd.DataFrame(pilot_rows)
display(pilot[['weather_day',*VARIABLES,'valid_24h_grid_coverage_fraction','negative_rain_pixel_fraction','hour_count']].round(4))
pilot_seq=pilot[VARIABLES].to_numpy(dtype=np.float32)
assert pilot_seq.shape==(14,4) and np.isfinite(pilot_seq).all()
assert pilot.weather_day.tolist()==pilot_days
print('PASS: Real Holombuwa weather sequence shape [14,4] with 14×24 hourly source records.')
print('Rainfall total over 14 days:',round(float(pilot['precipitation_mm'].sum()),3),'mm')
print('First period ends:',pilot.iloc[0].window_last_hour_utc,'UTC; newest:',pilot.iloc[-1].window_last_hour_utc,'UTC')
print('Potential review flags: negative rainfall pixel fraction > 0:',int((pilot.negative_rain_pixel_fraction>0).sum()),
      '| low grid coverage < 0.75:',int((pilot.valid_24h_grid_coverage_fraction<.75).sum()))
print('Cache sample:',cache_path(PILOT_STATION_ID,pilot_days[0]))

holombuwa: ERA5-Land query 2025-11-04 through 2025-11-08 (5 catchment-days)
holombuwa: ERA5-Land query 2025-11-09 through 2025-11-13 (5 catchment-days)
holombuwa: ERA5-Land query 2025-11-14 through 2025-11-17 (4 catchment-days)
holombuwa: 14 valid days; reused 0 cached days; new server calls 3.


,weather_day,precipitation_mm,temperature_C,dewpoint_C,surface_pressure_hPa,valid_24h_grid_coverage_fraction,negative_rain_pixel_fraction,hour_count
0,2025-11-04,0.2034,24.3652,21.3299,976.2761,1,0,24
1,2025-11-05,2.0724,24.1287,21.6784,975.4760,1,0,24
2,2025-11-06,5.0872,24.0166,21.4839,974.7392,1,0,24
3,2025-11-07,8.3017,23.5870,21.9369,974.9314,1,0,24
4,2025-11-08,4.3774,23.7878,21.6270,975.6004,1,0,24
5,2025-11-09,6.5003,24.0951,21.7391,975.5886,1,0,24
6,2025-11-10,0.9254,24.4099,21.2605,975.0575,1,0,24
7,2025-11-11,8.6067,23.6056,21.7640,975.6035,1,0,24
8,2025-11-12,1.5150,24.2620,21.9098,975.2819,1,0,24
9,2025-11-13,12.7244,24.2222,22.1958,975.2341,1,0,24


PASS: Real Holombuwa weather sequence shape [14,4] with 14×24 hourly source records.
Rainfall total over 14 days: 70.342 mm
First period ends: 2025-11-04T03:00:00 UTC; newest: 2025-11-17T03:00:00 UTC
Potential review flags: negative rainfall pixel fraction > 0: 0 | low grid coverage < 0.75: 0
Cache sample: C:\Users\ASUS\projects\rp_project\J26-DS-314\data\processed\c4\weather\cache\daily\holombuwa\2025-11-04.json


## Phase 2 — Deliberately process the remaining unique catchment-days, in small restartable batches

In [8]:
# Cell 7 — Bulk guard: inspect real pilot before turning this on
print('RUN_BULK:',RUN_BULK)
print('NEW stations processed at most per run:',MAX_NEW_STATIONS_PER_RUN)
print('Already fully cached stations:',sum(all(cache_path(sid,d).exists() for d in ds) for sid,ds in expected_days.items()), '/',len(expected_days))
if not RUN_BULK:
    print('SAFE: No full-dataset Earth Engine extraction will occur until RUN_BULK=True.')

RUN_BULK: True
NEW stations processed at most per run: 3
Already fully cached stations: 0 / 39


In [18]:
# Cell 8 — All labelled gauges: remote queries only for missing catchment-days
# Cell 8 deliberately does NOTHING if RUN_BULK=False. Safe to rerun after interruptions.
if not RUN_BULK:
    print('Bulk retrieval SKIPPED. Set RUN_BULK=True in Cell 1 only after pilot validation.')
else:
    done_before=[sid for sid,days in expected_days.items() if all(cache_path(sid,d).exists() for d in days)]
    pending=[sid for sid,days in expected_days.items() if sid not in done_before]
    selected=pending[:MAX_NEW_STATIONS_PER_RUN]
    print(f'Previously complete: {len(done_before)}/39; attempting {len(selected)} stations this run.')
    errors={}
    for sid in selected:
        try:
            ds,calls=get_weather_days(sid,expected_days[sid],allow_remote=True,verbose=True)
            assert len(ds)==len(expected_days[sid])
        except Exception as exc:
            errors[sid]=f'{type(exc).__name__}: {exc}'
            print('REVIEW / INCOMPLETE:',sid,errors[sid])
    ncomplete=sum(all(read_cached(sid,d) is not None for d in days) for sid,days in expected_days.items())
    print(f'Now completely cached: {ncomplete}/39 labelled gauges.')
    if errors:
        print('Errors are NOT treated as valid zero-weather observations. Fix and rerun missing days.')
        for sid,msg in errors.items(): print(sid,'—',msg[:450])
    if ncomplete<39:
        print('RERUN Cell 8 (or increase MAX_NEW_STATIONS_PER_RUN conservatively) until all 39 stations complete.')

Previously complete: 36/39; attempting 3 stations this run.
wellawaya: ERA5-Land query 2025-11-04 through 2025-11-08 (5 catchment-days)
wellawaya: ERA5-Land query 2025-11-09 through 2025-11-13 (5 catchment-days)
wellawaya: ERA5-Land query 2025-11-14 through 2025-11-18 (5 catchment-days)
wellawaya: ERA5-Land query 2025-11-19 through 2025-11-23 (5 catchment-days)
wellawaya: ERA5-Land query 2025-11-24 through 2025-11-28 (5 catchment-days)
wellawaya: ERA5-Land query 2025-11-29 through 2025-12-03 (5 catchment-days)
wellawaya: ERA5-Land query 2025-12-04 through 2025-12-08 (5 catchment-days)
wellawaya: ERA5-Land query 2025-12-09 through 2025-12-13 (5 catchment-days)
wellawaya: ERA5-Land query 2025-12-14 through 2025-12-18 (5 catchment-days)
wellawaya: ERA5-Land query 2025-12-19 through 2025-12-23 (5 catchment-days)
wellawaya: ERA5-Land query 2025-12-24 through 2025-12-25 (2 catchment-days)
wellawaya: 52 valid days; reused 0 cached days; new server calls 11.
weraganthota: ERA5-Land query 2025-

## Phase 3 — Assemble the 1,429 overlapping sequences from **local cache only**, verify, and export

In [19]:
# Cell 9 — Coverage QA, no GEE calls; don't fabricate zeros or impute missing weather
rows=[]; missing_pairs=[]
for sid,days in expected_days.items():
    for d in days:
        c=read_cached(sid,d)
        if c is None:
            missing_pairs.append((sid,d))
        else:
            rows.append(c)
daily=pd.DataFrame(rows)
assert daily[['station_id','weather_day']].duplicated().sum()==0 if len(daily) else True
complete_station_days=len(rows); expected_station_days=sum(map(len,expected_days.values()))
print(f'Weather catchment-days: {complete_station_days}/{expected_station_days} cached and validated.')
print('Missing station-days:',len(missing_pairs))
if missing_pairs: print('First missing:',missing_pairs[:8])
if len(daily):
    display(daily[VARIABLES+['valid_24h_grid_coverage_fraction','negative_rain_pixel_fraction']].describe().round(4))
    print('Daily rainfall < 0:',int((daily.precipitation_mm<0).sum()),
          '| > 300 mm:',int((daily.precipitation_mm>300).sum()),
          '| grid coverage <75%:',int((daily.valid_24h_grid_coverage_fraction<.75).sum()),
          '| any hourly negative pixel:',int((daily.negative_rain_pixel_fraction>0).sum()))
    extreme=daily.loc[(daily.precipitation_mm>300)|(daily.negative_rain_pixel_fraction>0)|(daily.valid_24h_grid_coverage_fraction<.75),
        ['station_id','weather_day','precipitation_mm','valid_24h_grid_coverage_fraction','negative_rain_pixel_fraction']]
    if len(extreme):
        print('Rows to review before treating the dataset as finalized:'); display(extreme.head(30))

Weather catchment-days: 2026/2026 cached and validated.
Missing station-days: 0


,precipitation_mm,temperature_C,dewpoint_C,surface_pressure_hPa,valid_24h_grid_coverage_fraction,negative_rain_pixel_fraction
count,2026.0000,2026.0000,2026.0000,2026.0000,2026.0,2026.0
mean,14.9711,23.3510,21.0001,964.6327,1.0,0.0
std,26.1048,1.6751,1.5350,31.8223,0.0,0.0
min,0.0004,17.4273,13.2479,863.0754,1.0,0.0
25%,1.6244,22.5446,20.2207,950.1475,1.0,0.0
50%,5.3138,23.7065,21.3731,975.0110,1.0,0.0
75%,16.6379,24.4479,22.1321,986.3695,1.0,0.0
max,227.4098,27.9384,23.6768,1000.6548,1.0,0.0


Daily rainfall < 0: 0 | > 300 mm: 0 | grid coverage <75%: 0 | any hourly negative pixel: 0


In [20]:
# Cell 10 — Save aligned [1429,14,4] and provenance only after ALL required daily weather is present.
# Safe to rerun: compares bytes and preserves source/candidate polygons without modification.
if missing_pairs:
    print('NOT EXPORTED: Missing local ERA5-Land catchment-days; finish Cell 8 first.')
else:
    daily=daily.sort_values(['station_id','weather_day']).reset_index(drop=True)
    daily_index={(r['station_id'],r['weather_day']):r for r in daily.to_dict('records')}
    aligned=[]; meta=[]
    for r in index.itertuples(index=False):
        days=days_for_forecast(r.forecast_day)
        vals=np.array([[float(daily_index[(r.station_id,d)][var]) for var in VARIABLES] for d in days],dtype=np.float32)
        assert vals.shape==(14,4) and np.isfinite(vals).all(),r.record_id
        assert days[-1]==r.forecast_day.date().isoformat()
        assert iso_ee(source_window_for_day(days[-1])[1]-timedelta(hours=1)) <             pd.Timestamp(r.forecast_day).strftime('%Y-%m-%d')+'T03:30:00', 'Invalid UTC comparison boundary'
        aligned.append(vals)
        meta.append({'record_id':r.record_id,'station_id':r.station_id,'forecast_date':r.forecast_day.date().isoformat(),
                     'split':r.split,'weather_available':True,'earliest_weather_day':days[0],
                     'last_weather_day':days[-1], 'weather_sequence_shape':'14x4',
                     'weather_source':ASSET,'weather_last_hour_local':'08:30 Asia/Colombo'})
    X=np.stack(aligned,axis=0)
    assert X.shape==(1429,14,4) and np.isfinite(X).all()
    qa=daily[['station_id','weather_day','hour_count','valid_24h_grid_coverage_fraction',
        'negative_rain_pixel_fraction','window_start_utc','window_last_hour_utc','cache_key']].copy()
    summary=pd.DataFrame(meta)
    station_hashes={sid:polygon_sha[sid] for sid in sorted(polygon_sha)}
    source_config_hash = hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()
    manifest={
        'description':'C4 preliminary retrospective ERA5-Land weather; local 14x4 sequences and candidate-polygon daily stats',
        'source_config':CONFIG,'source_config_sha256':source_config_hash,
        'weather_feature_order':VARIABLES,'n_sequences':len(index),'n_gauges':len(expected_days),
        'n_unique_gauge_days':len(daily),'shape':[1429,14,4],
        'sequence_order':'14 historical 24-hour periods, oldest to newest; final hourly validity at 08:30 Asia/Colombo',
        'forecast_origin':'Day T at simulated 09:00 Asia/Colombo; publication latency not operationally verified',
        'polygon_sha256':station_hashes,'target_free_index_sha256':hashlib.sha256(INDEX_PATH.read_bytes()).hexdigest(),
        'weather_units':{'precipitation_mm':'mm/24h','temperature_C':'C','dewpoint_C':'C','surface_pressure_hPa':'hPa'},
        'imputation':'none; complete input coverage required',
        'spatial_note':'Native ERA5-Land grid ~11 km; default Earth Engine area-overlap-weighted mean; not subcatchment-resolved rainfall',
        'qa_limits':'Missing/negative/implausible values rejected during cache validation. Nonzero negative hourly pixel flag requires review.'
    }
    output_daily=WEATHER_DIR/'c4_weather_daily_era5land_PRIVATE.csv'
    output_qa=WEATHER_DIR/'c4_weather_QA_PRIVATE.csv'
    output_index=WEATHER_DIR/'c4_weather_sequence_index_1429_PRIVATE.csv'
    output_npz=WEATHER_DIR/'c4_weather_sequences_1429_PRIVATE.npz'
    output_manifest=WEATHER_DIR/'c4_weather_manifest_PRIVATE.json'
    print('Daily:',save_if_changed(output_daily,daily.to_csv(index=False,float_format='%.10g')))
    print('QA:',save_if_changed(output_qa,qa.to_csv(index=False,float_format='%.10g')))
    print('Sequence index:',save_if_changed(output_index,summary.to_csv(index=False)))
    # NumPy NPZ compressed; compare content hashes first to avoid rewriting identical training arrays.
    if output_npz.exists():
        with np.load(output_npz,allow_pickle=False) as existing:
            same=(np.array_equal(existing['X_weather'],X) and
                  np.array_equal(existing['record_id'],index.record_id.to_numpy(dtype=str)) and
                  np.array_equal(existing['feature_names'],np.asarray(VARIABLES,dtype=str)))
    else: same=False
    if same: print('Arrays: UNCHANGED')
    else:
        b=io.BytesIO();np.savez_compressed(b,X_weather=X, record_id=index.record_id.to_numpy(dtype=str),
             station_id=index.station_id.to_numpy(dtype=str), feature_names=np.asarray(VARIABLES,dtype=str))
        print('Arrays:',save_if_changed(output_npz,b.getvalue()))
    manifest['outputs_sha256']={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in [output_daily,output_qa,output_index,output_npz]}
    print('Manifest:',save_if_changed(output_manifest,json.dumps(manifest,indent=2,sort_keys=True)+'\n'))
    print('PASS: Local weather matrix [1429,14,4] saved in the exact Notebook 01 manifest order.')
    print('NO model training scaler was fitted; fit feature scaling on TRAIN ONLY in Notebook 07.')

Daily: SAVED
QA: SAVED
Sequence index: SAVED
Arrays: SAVED
Manifest: SAVED
PASS: Local weather matrix [1429,14,4] saved in the exact Notebook 01 manifest order.
NO model training scaler was fitted; fit feature scaling on TRAIN ONLY in Notebook 07.
